Checking PSD of R for DINO v3 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [ ]:
# download images
!mkdir -p coco
%cd coco
!wget http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip
%cd ..

/content/coco
--2026-03-28 11:11:37--  http://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.216.39.81, 52.216.86.163, 16.15.191.66, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.216.39.81|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 815585330 (778M) [application/zip]
Saving to: ‘val2017.zip’

val2017.zip         100%[===================>] 777.80M  18.3MB/s    in 46s     

2026-03-28 11:12:23 (17.0 MB/s) - ‘val2017.zip’ saved [815585330/815585330]

/content


In [ ]:
!pip install torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.7 MB/s eta 0:00:00


In [ ]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import torchmetrics
import cv2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
def make_transform(resize_size = 768):
    #to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([resize, normalize])

In [ ]:
img_size = 768
transform = make_transform(img_size)

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git

Cloning into 'dinov3'...
remote: Enumerating objects: 579, done.
remote: Counting objects: 100% (445/445), done.
remote: Compressing objects: 100% (337/337), done.
remote: Total 579 (delta 234), reused 108 (delta 108), pack-reused 134 (from 2)
Receiving objects: 100% (579/579), 12.44 MiB | 15.48 MiB/s, done.
Resolving deltas: 100% (248/248), done.


In [ ]:
!cp dinov3/hubconf.py .

In [ ]:
import sys
sys.path.append("dinov3")

In [ ]:
!wget http://www.agentspace.org/download/mydinov3.zip
!unzip -P dino mydinov3.zip

--2026-03-28 11:12:57--  http://www.agentspace.org/download/mydinov3.zip
Resolving www.agentspace.org (www.agentspace.org)... 62.168.101.9
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.agentspace.org/download/mydinov3.zip [following]
--2026-03-28 11:12:58--  https://www.agentspace.org/download/mydinov3.zip
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80324355 (77M) [application/zip]
Saving to: ‘mydinov3.zip’

mydinov3.zip        100%[===================>]  76.60M  16.6MB/s    in 5.7s    

2026-03-28 11:13:05 (13.5 MB/s) - ‘mydinov3.zip’ saved [80324355/80324355]

Archive:  mydinov3.zip
  inflating: dinov3_vits16_pretrain_lvd1689m.pth  


In [ ]:
backbone = torch.hub.load('.', 'dinov3_vits16', source='local', weights='dinov3_vits16_pretrain_lvd1689m.pth') # dinov3_vits16

Downloading: "file:///content/dinov3_vits16_pretrain_lvd1689m.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vits16_pretrain_lvd1689m.pth


100%|██████████| 82.5M/82.5M [00:00<00:00, 1.66GB/s]


In [ ]:
backbone.eval()

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm): LayerN

In [ ]:
import inspect
print(inspect.getsource(backbone.__class__))

class DinoVisionTransformer(nn.Module):
    def __init__(
        self,
        *,
        img_size: int = 224,
        patch_size: int = 16,
        in_chans: int = 3,
        pos_embed_rope_base: float = 100.0,
        pos_embed_rope_min_period: float | None = None,
        pos_embed_rope_max_period: float | None = None,
        pos_embed_rope_normalize_coords: Literal["min", "max", "separate"] = "separate",
        pos_embed_rope_shift_coords: float | None = None,
        pos_embed_rope_jitter_coords: float | None = None,
        pos_embed_rope_rescale_coords: float | None = None,
        pos_embed_rope_dtype: str = "bf16",
        embed_dim: int = 768,
        depth: int = 12,
        num_heads: int = 12,
        ffn_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_path_rate: float = 0.0,
        layerscale_init: float | None = None,
        norm_layer: str = "layernorm",
        ffn_layer: str = "mlp",
        ffn_bias: bool = True,
        proj_bias: bool = True,
 

In [ ]:
test_dir = "coco/val2017"
files = os.listdir(test_dir)
len(files)

5000

In [ ]:
features = []
mn = 1e9
backbone.to(device).to(torch.bfloat16)
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    blob = cv2.dnn.blobFromImage(img,1.0/255)
    with torch.inference_mode():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            inp = transform(torch.tensor(blob[0])).unsqueeze(0).to(device)
            #out = backbone.forward_features(inp)
            out = backbone.get_intermediate_layers(inp,n=[11],norm=False)[0]

    #feats = out['x_prenorm'][0][1+backbone.n_storage_tokens:].view(48,48,-1)
    feats = out[0].view(48,48,-1)
    features.append(feats)
    flat = feats.view(-1, feats.shape[-1])  # shape: (H*W, C)
    W = flat @ flat.T
    minimum = W.min().item()
    if minimum < mn:
        mn = minimum
        print(i,minimum, "!!!!!!" if minimum <= 0 else "")

0 161792.0 
2 151552.0 
6 146432.0 
391 139264.0 
1030 135168.0 
1548 132096.0 


In [ ]:
features = torch.stack(features)

In [ ]:
features.shape

torch.Size([5000, 48, 48, 384])

In [ ]:
features.min(),features.max()

(tensor(-222., device='cuda:0', dtype=torch.bfloat16),
 tensor(548., device='cuda:0', dtype=torch.bfloat16))

The values of W are always positive, d is well-defined, and R is PSD for all samples